<a href="https://colab.research.google.com/github/LAB-FAM/nice-rag-project/blob/main/colab/query_understanding_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# INSTALL REQUIRED LIBRARIES
!pip install -qU langgraph langchain langchain-community langchain-ollama chromadb pydantic sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.2/168.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 93.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 114.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/

In [4]:
# SETUP ENVIRONMENT (Vector DB Extraction)
import os
import shutil
import zipfile

# 1. Delete previous folders to ensure a clean run
for folder in ["methodology_db", "vector_db"]:
    if os.path.exists(folder):
        shutil.rmtree(folder)

# 2. Extract vector_db.zip to vector_db folder (Phase 0 output)
if os.path.exists("vector_db.zip"):
    print("[*] Extracting vector_db.zip...")
    with zipfile.ZipFile("vector_db.zip", 'r') as zip_ref:
        zip_ref.extractall("vector_db")
    print("[*] Extraction complete.")
else:
    print("[!] Warning: vector_db.zip not found. Make sure you uploaded it.")

[*] Extracting vector_db.zip...
[*] Extraction complete.


In [5]:
# INSTALL AND RUN OLLAMA IN BACKGROUND

!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time
print("[*] Starting Ollama server...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# We will use llama3 as our local reasoning agent
print("[*] Pulling llama3 model (This will take a few minutes)...")
!ollama pull llama3
print("[*] Pulling nomic-embed-text model...")
!ollama pull nomic-embed-text
print("[*] Setup complete!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
[*] Starting Ollama server

In [6]:
# LANGGRAPH NODE 1 IMPLEMENTATION (ADVANCED RAG)
import json
from typing import TypedDict, List
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma

# 1. UNIFIED STATE DEFINITION
class UnifiedGraphState(TypedDict):
    research_question: str
    primary_condition: str
    related_conditions: List[str]
    observations: List[str]
    concept_type: str
    snomed_top_hierarchy: str
    search_terms: List[str]
    explicit_exclusions: List[str]
    relevant_guidelines: List[str]
    suggested_validation_sources: List[str]
    ambiguity_notes: str
    qof_domain_prefix: str
    qof_rules_text: str
    candidate_codes: List[dict]
    final_codes: List[dict]

In [7]:
# 2. LLM STRUCTURED OUTPUT SCHEMA (PYDANTIC)
class QueryExtraction(BaseModel):
    primary_condition: str = Field(description="The main disease or condition (e.g., 'Type 2 Diabetes')")
    related_conditions: List[str] = Field(description="Other mentioned conditions", default_factory=list)
    observations: List[str] = Field(description="CRITICAL: Extract any tests, measurements, or biomarkers mentioned.", default_factory=list)
    concept_type: str = Field(description="SNOMED concept type (e.g., 'disease', 'finding', 'procedure')", default="")
    snomed_top_hierarchy: str = Field(description="Top level SNOMED hierarchy category", default="")
    search_terms: List[str] = Field(description="List of optimized search synonyms", default_factory=list)
    explicit_exclusions: List[str] = Field(description="Conditions or keywords that must be explicitly excluded", default_factory=list)
    qof_domain_prefix: str = Field(description="The standard UK QOF domain abbreviation (e.g., 'DM', 'HYP', 'OB'). If unknown, leave empty.", default="")
    suggested_validation_sources: List[str] = Field(description="Suggested sources for validation", default_factory=list)
    ambiguity_notes: str = Field(description="Any missing or ambiguous info", default="")

In [8]:
# 3. INITIALIZE MODELS & VECTOR DATABASE
print("[*] Initializing local models & connecting to ChromaDB...")
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vector_db = Chroma(persist_directory="vector_db/methodology_db", embedding_function=embeddings)

llm = ChatOllama(model="llama3", temperature=0)
structured_llm = llm.with_structured_output(QueryExtraction)

[*] Initializing local models & connecting to ChromaDB...


/tmp/ipykernel_4356/430667074.py:4: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(persist_directory="vector_db/methodology_db", embedding_function=embeddings)


In [9]:
# --- FILE MAPPING DICTIONARY ---
# This maps the extracted clinical conditions to their exact PDF filenames
# It's crucial for Dynamic Metadata Filtering to block irrelevant guidelines (like Hypertension)
CONDITION_TO_FILE = {
    "Type 2 Diabetes Mellitus": "NG28_type_2_diabetes.pdf",
    "Obesity": "NG246_obesity.pdf",
    "Hypertension": "NG136_hypertension.pdf"
}

QOF_FILE = "qof_combined.pdf" # QOF should always be accessible
CHROMA_DB_DIR = "vector_db/methodology_db"
EMBEDDING_MODEL_NAME = "nomic-embed-text"

In [10]:
def clean_boilerplate_text(text):
    """
    Cleans up repetitive boilerplate texts, headers, and footers from NICE guidelines
    so they don't distract the LLM during the generation phase.
    """
    # List of noisy strings to remove
    noise = [
        "© NICE 2026. All rights reserved.",
        "Subject to Notice of rights",
        "(https://www.nice.org.uk/terms-and- conditions#notice-of-rights)",
        "(https://www.nice.org.uk/terms-and-conditions#notice-of-rights)",
        "Return to recommendations",
        "Why the committee made the recommendations",
        "How the recommendations might affect practice"
    ]
    cleaned_text = text
    for n in noise:
        cleaned_text = cleaned_text.replace(n, "")

    # Simple logic to remove stray "Page X of Y" lines
    lines = [line for line in cleaned_text.split('\n') if not line.strip().startswith("Page ") and "of 1" not in line]
    return " ".join(lines).strip()

In [11]:
def get_relevant_pdf(condition_name, pdf_folder="pdf_docs"):
    """
    SMART DOCUMENT ROUTER:
    First checks the exact dictionary mapping. If not found, it dynamically scans
    the pdf_docs directory to find a matching filename, preventing future bottlenecks.
    """
    if not condition_name:
        return None

    # 1. Fast Track: Exact Match in Dictionary
    if condition_name in CONDITION_TO_FILE:
        return CONDITION_TO_FILE[condition_name]

    # 2. Dynamic Fallback: Scan the folder for filenames containing the condition words
    print(f"[*] '{condition_name}' not in dictionary. Scanning {pdf_folder} dynamically...")
    if not os.path.exists(pdf_folder):
        return None

    # Convert condition to lowercase words for fuzzy matching (e.g. "heart failure")
    search_terms = condition_name.lower().replace("-", " ").split()

    for filename in os.listdir(pdf_folder):
        if filename.endswith(".pdf") and filename != QOF_FILE:
            name_lower = filename.lower().replace("_", " ")
            # If the main keywords of the condition are in the filename, we assume a match
            if any(term in name_lower for term in search_terms if len(term) > 4):
                print(f"[*] Auto-mapped '{condition_name}' to '{filename}' based on filename.")
                return filename

    return None

In [15]:
def advanced_retrieval(extracted_info, vector_db):
    """
    Takes the structured output from Node 1 (Query Understanding)
    and performs a highly filtered, MMR-based search on ChromaDB.
    """
    print("\n--- [NODE 1 - Phase 2] ADVANCED RAG RETRIEVAL STARTED ---")

    # Safely extract values whether extracted_info is a dict or a Pydantic object
    if isinstance(extracted_info, dict):
        primary = extracted_info.get('primary_condition', '')
        related_conditions = extracted_info.get('related_conditions', [])
        qof_prefix = extracted_info.get('qof_domain_prefix', '')
        target_demographic = extracted_info.get('target_demographic', 'adult').lower()
        exclusions = extracted_info.get('explicit_exclusions', [])
    else:
        primary = getattr(extracted_info, 'primary_condition', '')
        related_conditions = getattr(extracted_info, 'related_conditions', [])
        qof_prefix = getattr(extracted_info, 'qof_domain_prefix', '')
        target_demographic = getattr(extracted_info, 'target_demographic', 'adult').lower()
        exclusions = getattr(extracted_info, 'explicit_exclusions', [])

    # Build allowed files list for CLINICAL searches using the SMART ROUTER
    clinical_allowed_files = []

    primary_pdf = get_relevant_pdf(primary)
    if primary_pdf:
        clinical_allowed_files.append(f"pdf_docs/{primary_pdf}")

    for related in related_conditions:
        related_pdf = get_relevant_pdf(related)
        if related_pdf and f"pdf_docs/{related_pdf}" not in clinical_allowed_files:
            clinical_allowed_files.append(f"pdf_docs/{related_pdf}")

    print(f"[*] Applying strict clinical metadata filter. Allowed files: {clinical_allowed_files}")
    clinical_search_filter = {"source": {"$in": clinical_allowed_files}}

    # 2. ISOLATED SUB-QUERY GENERATION & EXECUTION
    all_retrieved_documents = []

    # Set the dynamic prefix based on demographic
    demo_prefix = "Pediatric" if target_demographic == "pediatric" else "Adult"

    # --- QUERY 1: QOF Specific Query ---
    if primary:
        qof_query = f"QOF indicator business rules register {qof_prefix} {primary}"
        print(f"    -> Querying Vector DB: '{qof_query}' (Strict QOF File Filter - Similarity Only)")
        docs = vector_db.similarity_search(
            qof_query,
            k=3,
            filter={"source": f"pdf_docs/{QOF_FILE}"}
        )
        all_retrieved_documents.extend(docs)

        # --- QUERY 2: Clinical Definitions & Management (Primary) ---
        clin_query = f"{demo_prefix} Definitions, diagnostic criteria, clinical management rules, pharmacological treatment {primary}"
        print(f"    -> Querying Vector DB: '{clin_query}' (Clinical Filter)")
        docs = vector_db.max_marginal_relevance_search(
            clin_query,
            k=3,
            fetch_k=15,
            lambda_mult=0.5,
            filter=clinical_search_filter if clinical_allowed_files else None
        )
        all_retrieved_documents.extend(docs)

    # --- QUERY 3: Related Conditions (e.g., Obesity) ---
    for related in related_conditions:
        rel_query = f"{demo_prefix} Diagnostic criteria clinical management {related}"
        print(f"    -> Querying Vector DB: '{rel_query}' (Clinical Filter)")
        docs = vector_db.max_marginal_relevance_search(
            rel_query,
            k=3,
            fetch_k=15,
            lambda_mult=0.5,
            filter=clinical_search_filter if clinical_allowed_files else None
        )
        all_retrieved_documents.extend(docs)

    # 4. DEDUPLICATION, CLEANING, AND DYNAMIC SHIELDING
    print("[*] Deduplicating, cleaning, and applying dynamic demographic shields...")
    unique_chunks = {}
    for doc in all_retrieved_documents:
        content_lower = doc.page_content.lower()
        source_file = doc.metadata.get('source', '')

        # FIX: QOF DOMAIN SHIELD
        if "qof_combined" in source_file and qof_prefix:
            if qof_prefix.lower() not in content_lower:
                continue

        # FIX: DYNAMIC DEMOGRAPHIC SHIELD
        has_pediatric_terms = any(term in content_lower for term in ["children", "young person", "paediatric", "pediatric", "child", "infant"])
        has_adult_terms = "adult" in content_lower

        if target_demographic == "pediatric":
            # If the patient is a child, block pure adult chunks
            if has_adult_terms and not has_pediatric_terms:
                continue
        else:
            # If the patient is an adult, block pure pediatric chunks
            if has_pediatric_terms and not has_adult_terms:
                continue

        # FIX: EXPLICIT EXCLUSION SHIELD
        # Drop chunks that contain the exact exclusion phrases to prevent context pollution
        should_exclude = False
        if exclusions:
            for exc in exclusions:
                if exc.strip() and exc.lower() in content_lower:
                    should_exclude = True
                    break

        if should_exclude:
            print(f"    [!] Shield activated: Chunk discarded due to exclusion rule.")
            continue

        doc_snippet = doc.page_content[:100]
        if doc_snippet not in unique_chunks:
            cleaned_content = clean_boilerplate_text(doc.page_content)
            doc.page_content = cleaned_content
            unique_chunks[doc_snippet] = doc

    final_documents = list(unique_chunks.values())

    print(f"--- [NODE 1 - Phase 2] COMPLETE: Retrieved {len(final_documents)} highly relevant, {target_demographic}-focused clean chunks ---")
    return final_documents

In [16]:
def run_rag_pipeline(state: 'UnifiedGraphState') -> 'UnifiedGraphState':
    print("\n--- [NODE 1 - Phase 1] QUERY UNDERSTANDING STARTED ---")

    query = state["research_question"]
    print(f"[*] Processing Query: '{query}'")

    # ADDED target_demographic instruction to the LLM prompt
    system_prompt = """You are an expert clinical data analyst for the NHS. Extract medical entities from the query.
    CRITICAL: Deduce the correct UK QOF domain prefix (e.g., 'DM' for Diabetes, 'HYP' for Hypertension, 'OB' for Obesity).
    CRITICAL: Detect the target demographic from the query. Output strictly 'adult', 'pediatric', or 'general'.

    Example Output:
    {{
        "primary_condition": "Type 2 Diabetes Mellitus",
        "related_conditions": ["Obesity"],
        "observations": ["Body mass index (BMI)"],
        "explicit_exclusions": ["Suspected cases", "Gestational diabetes"],
        "search_terms": ["Type 2 diabetes", "T2DM"],
        "qof_domain_prefix": "DM",
        "concept_type": "disease",
        "snomed_top_hierarchy": "Clinical finding",
        "target_demographic": "adult"
    }}
    """
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("user", "{question}")
    ])

    chain = prompt_template | structured_llm
    extracted_data = chain.invoke({"question": query})

    primary_cond = getattr(extracted_data, 'primary_condition', '')
    qof_prefix = getattr(extracted_data, 'qof_domain_prefix', '')
    exclusions = getattr(extracted_data, 'explicit_exclusions', [])
    target_demo = getattr(extracted_data, 'target_demographic', 'adult')

    print(f"[*] Primary Condition: {primary_cond} (QOF: {qof_prefix})")
    print(f"[*] Target Demographic: {target_demo}")
    print(f"[*] Exclusions Found: {exclusions}")

    final_docs = advanced_retrieval(extracted_data, vector_db)

    qof_rules_payload = ""
    found_guidelines = set()

    if not final_docs:
        print("[!] No documents retrieved.")
    else:
        for i, doc in enumerate(final_docs, 1):
            source = os.path.basename(doc.metadata.get('source', 'Unknown'))
            filename = source.replace('.pdf', '')
            found_guidelines.add(filename)

            # print(f"\n--- Document {i} (Source: {source}) ---")
            # content_preview = doc.page_content[:800]
            # print(f"{content_preview}...\n")

            qof_rules_payload += f"--- Source: {filename} ---\n"
            qof_rules_payload += doc.page_content + "\n\n"


    state.update({
        "primary_condition": primary_cond,
        "related_conditions": getattr(extracted_data, 'related_conditions', []),
        "observations": getattr(extracted_data, 'observations', []),
        "explicit_exclusions": exclusions,
        "search_terms": getattr(extracted_data, 'search_terms', []),
        "qof_domain_prefix": qof_prefix,
        "concept_type": getattr(extracted_data, 'concept_type', ''),
        "snomed_top_hierarchy": getattr(extracted_data, 'snomed_top_hierarchy', ''),
        "suggested_validation_sources": getattr(extracted_data, 'suggested_validation_sources', []),
        "ambiguity_notes": getattr(extracted_data, 'ambiguity_notes', ""),

        "qof_rules_text": qof_rules_payload.strip(),
        "relevant_guidelines": list(found_guidelines),

        "candidate_codes": state.get("candidate_codes", []) if hasattr(state, "get") else getattr(state, "candidate_codes", []),
        "final_codes": state.get("final_codes", []) if hasattr(state, "get") else getattr(state, "final_codes", [])
    })

    print("--- [NODE 1 - Phase 1] COMPLETE ---")
    return state

In [17]:
# MOCK TEST
if __name__ == "__main__":
    initial_state = {
        "research_question": "Patient has a history of Type 2 Diabetes and is currently suffering from Obesity. Find the relevant active SNOMED codes. Exclude suspected cases.",
        "primary_condition": "", "related_conditions": [], "observations": [],
        "search_terms": [], "explicit_exclusions": [], "qof_domain_prefix": "",
        "concept_type": "", "snomed_top_hierarchy": "", "suggested_validation_sources": [],
        "ambiguity_notes": "", "qof_rules_text": "", "relevant_guidelines": [],
        "candidate_codes": [], "final_codes": []
    }

    result = run_rag_pipeline(initial_state)

    print("\n=== FINAL STATE SUMMARY ===")
    print(f"Primary Condition: {result['primary_condition']} (QOF: {result['qof_domain_prefix']})")
    print(f"Related Conditions: {result['related_conditions']}")
    print(f"Observations: {result['observations']}")
    print(f"Exclusions: {result['explicit_exclusions']}")
    print(f"Guidelines Cited: {result['relevant_guidelines']}")
    print(f"\nExtracted QOF Rules Payload:\n{result['qof_rules_text']}...")


--- [NODE 1 - Phase 1] QUERY UNDERSTANDING STARTED ---
[*] Processing Query: 'Patient has a history of Type 2 Diabetes and is currently suffering from Obesity. Find the relevant active SNOMED codes. Exclude suspected cases.'
[*] Primary Condition: Type 2 Diabetes Mellitus (QOF: DM)
[*] Target Demographic: adult
[*] Exclusions Found: ['Suspected cases']

--- [NODE 1 - Phase 2] ADVANCED RAG RETRIEVAL STARTED ---
[*] Applying strict clinical metadata filter. Allowed files: ['pdf_docs/NG28_type_2_diabetes.pdf', 'pdf_docs/NG246_obesity.pdf']
    -> Querying Vector DB: 'QOF indicator business rules register DM Type 2 Diabetes Mellitus' (Strict QOF File Filter - Similarity Only)
    -> Querying Vector DB: 'Adult Definitions, diagnostic criteria, clinical management rules, pharmacological treatment Type 2 Diabetes Mellitus' (Clinical Filter)
    -> Querying Vector DB: 'Adult Diagnostic criteria clinical management Obesity' (Clinical Filter)
[*] Deduplicating, cleaning, and applying dynamic de